# Task 5 — Explainability (Grad-CAM + SHAP)
**COMP41840 AI for Health**  
**Owner:** Sergio  
**Goal:** Explain the imaging model via Grad-CAM and the tabular model via SHAP values

> **Prerequisite:** Run notebooks 02 and 03 first


In [ ]:
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import shap
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torchvision import transforms
import xgboost as xgb
from monai.networks.nets import DenseNet121

sns.set_theme(style='whitegrid')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT   = Path('../data/raw')
RESULTS_DIR = Path('../results')
(RESULTS_DIR / 'figures').mkdir(exist_ok=True)


def iter_class_images(data_root: Path, cls: str):
    paths = []
    d = data_root / cls / 'images'
    if d.exists():
        paths.extend(sorted(d.glob('*.png')))
    for sub in sorted((data_root / 'dataset').glob(f'dataset*/{cls}/images')):
        paths.extend(sorted(sub.glob('*.png')))
    return paths

---
## Part A — Grad-CAM for Imaging Model

### A.1 — Load trained imaging model

In [ ]:
model = DenseNet121(spatial_dims=2, in_channels=3, out_channels=2).to(DEVICE)
model.load_state_dict(torch.load(RESULTS_DIR / 'best_imaging_model.pt', map_location=DEVICE))
model.eval()
print('Model loaded.')

### A.2 — Grad-CAM Implementation

In [ ]:
class GradCAM:
    """Minimal Grad-CAM for a MONAI DenseNet121."""

    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, x, class_idx=None):
        self.model.zero_grad()
        out = self.model(x)
        if class_idx is None:
            class_idx = out.argmax(dim=1).item()
        score = out[0, class_idx]
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)   # GAP
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx


# Spatial feature maps: last _DenseBlock before final BatchNorm in MONAI DenseNet121
ft = model.features
target_layer = ft[-2] if isinstance(ft[-1], nn.BatchNorm2d) else ft[-1]
grad_cam = GradCAM(model, target_layer)

### A.3 — Visualise Grad-CAM for Sample Images

In [ ]:
IMG_SIZE = 224
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

def visualise_gradcam(img_path, label_str, save_path=None):
    raw_img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    tensor  = transform(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    tensor.requires_grad_()

    cam, pred_cls = grad_cam(tensor)
    heatmap = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    heatmap_color = cm.jet(heatmap)[:, :, :3]
    overlay = 0.5 * np.array(raw_img) / 255 + 0.5 * heatmap_color

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(raw_img, cmap='gray'); axes[0].set_title(f'Original ({label_str})')
    axes[1].imshow(heatmap, cmap='jet'); axes[1].set_title('Grad-CAM Heatmap')
    axes[2].imshow(overlay); axes[2].set_title(f'Overlay | Pred: {["benign","malignant"][pred_cls]}')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


mal_paths = iter_class_images(DATA_ROOT, 'malignant')
ben_paths = iter_class_images(DATA_ROOT, 'benign')
if not mal_paths or not ben_paths:
    raise FileNotFoundError('No sample images found for Grad-CAM. Extract Kaggle data under data/raw/.')
mal_sample = mal_paths[0]
ben_sample = ben_paths[0]

visualise_gradcam(mal_sample, 'malignant', RESULTS_DIR / 'figures/gradcam_malignant.png')
visualise_gradcam(ben_sample, 'benign',    RESULTS_DIR / 'figures/gradcam_benign.png')

### A.4 — Grad-CAM Discussion

Describe what regions the model attends to:
- For malignant cases: does the heatmap centre on the tumour mass?
- For benign cases: are the highlighted regions clinically plausible?
- Discuss any surprising or concerning attention patterns.

---
## Part B — SHAP Values for Tabular Model

### B.1 — Load Trained XGBoost Model & Preprocess

In [ ]:
# Load tabular model + imputed matrices saved by notebook 03
pre_path = RESULTS_DIR / 'tabular_preprocess.joblib'
model_path = RESULTS_DIR / 'tabular_xgb_model.json'
if not pre_path.exists() or not model_path.exists():
    raise FileNotFoundError(
        'Run notebooks/03_tabular_model.ipynb first so it writes '
        'tabular_preprocess.joblib and tabular_xgb_model.json'
    )

pre = joblib.load(pre_path)
feature_cols = pre['feature_cols']
X_train_imp = pre['X_train_imp']
X_val_imp = pre['X_val_imp']
X_test_imp = pre['X_test_imp']
y_train = pre['y_train']
y_val = pre['y_val']
y_test = pre['y_test']

X_bg = np.vstack([X_train_imp, X_val_imp])
y_bg = np.concatenate([y_train, y_val])
X_imp_df = pd.DataFrame(X_bg, columns=feature_cols)
y = y_bg

xgb_model = xgb.XGBClassifier()
xgb_model.load_model(str(model_path))
print('XGBoost + tabular matrices loaded for SHAP.')

X_test_df = pd.DataFrame(X_test_imp, columns=feature_cols)

### B.2 — SHAP Summary (Beeswarm) Plot

In [ ]:
max_vis = min(500, len(X_imp_df))
X_vis = X_imp_df.iloc[:max_vis]
# Model-only explainer avoids masker paths that pull optional Transformers deps on some installs
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_vis)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_vis, show=False)
plt.title('SHAP summary (malignant logit) - train+val sample')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bar summary — mean absolute SHAP
plt.figure(figsize=(9, 6))
shap.summary_plot(sv, X_vis, plot_type='bar', show=False)
plt.title('SHAP feature importance (mean |SHAP|), malignant vs benign')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

### B.3 — SHAP Waterfall for Individual Patients

In [ ]:
# Per-patient SHAP bar plots (test set) — robust across SHAP versions
explainer_test = shap.TreeExplainer(xgb_model)
sv_test = explainer_test.shap_values(X_test_df)
sv1 = sv_test[1] if isinstance(sv_test, list) else sv_test

malignant_idxs = np.where(y_test == 1)[0]
benign_idxs = np.where(y_test == 0)[0]
if len(malignant_idxs) == 0 or len(benign_idxs) == 0:
    print('Need both classes in test set for paired SHAP plots.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, idx, title in [
        (axes[0], malignant_idxs[0], 'Test — malignant'),
        (axes[1], benign_idxs[0], 'Test — benign'),
    ]:
        row = sv1[idx]
        order = np.argsort(np.abs(row))[-18:]
        ax.barh(np.array(feature_cols)[order], row[order], color='steelblue')
        ax.axvline(0, color='k', lw=0.5)
        ax.set_title(title)
    plt.suptitle('SHAP values (top features) for example test patients')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'figures/shap_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()

### B.4 — SHAP Dependence Plots for Top Features

In [ ]:
mean_abs = np.abs(sv).mean(0)
top2 = np.argsort(mean_abs)[-2:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, feat_idx in zip(axes, top2):
    feat_name = feature_cols[feat_idx]
    shap.dependence_plot(feat_name, sv, X_vis, ax=ax, show=False)
    ax.set_title(f'SHAP dependence: {feat_name}')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

### B.5 — Explainability Discussion

Answer the following questions in the report:

1. **Grad-CAM:** Do the highlighted image regions align with known ultrasound indicators of malignancy (e.g., irregular margins, posterior shadowing)? Are there spurious correlations?
2. **SHAP:** Which clinical features drive malignant predictions? Are they consistent with domain knowledge?
3. **Plausibility:** Do both methods provide clinically plausible explanations, or do they reveal model artefacts?
4. **Limitations:** What are the limits of Grad-CAM and SHAP in a medical context?